# Sentiment analysis

**TP2 – année choisie : 1960**

Nous gardons les deux outils du cours : TextBlob-FR et un modèle BERT entraîné sur des critiques de films (Allociné). Nous ajoutons l'analyse de 10 phrases tirées des journaux de 1960, puis un tableau qui résume leur polarité et leur subjectivité.

## 1. Textblob-FR

Documentation: https://textblob.readthedocs.io/en/dev/

### Imports

In [1]:
import pandas as pd
from textblob import Blobber
from textblob_fr import PatternTagger, PatternAnalyzer

### Création d'une fonction `get_sentiment`

In [2]:
tb = Blobber(pos_tagger=PatternTagger(), analyzer=PatternAnalyzer())

def get_sentiment(input_text):
    blob = tb(input_text)
    polarity, subjectivity = blob.sentiment
    polarity_perc = f"{100*abs(polarity):.0f}"
    subjectivity_perc = f"{100*subjectivity:.0f}"
    if polarity > 0:
        polarity_str = f"{polarity_perc}% positive"
    elif polarity < 0:
        polarity_str = f"{polarity_perc}% negative"
    else:
        polarity_str = "neutral"
    if subjectivity > 0:
        subjectivity_str = f"{subjectivity_perc}% subjective"
    else:
        subjectivity_str = "perfectly objective"
    print(f"This text is {polarity_str} and {subjectivity_str}.")

### Analyser le sentiment d'une phrase

In [3]:
get_sentiment("Ce journal est vraiment super intéressant.")

This text is 65% positive and 75% subjective.


In [4]:
get_sentiment("Cette phrase est négative et je ne suis pas content !")

This text is 41% negative and 60% subjective.


### Analyser 10 phrases tirées des journaux de 1960

Nous avons choisi les phrases selon trois critères :
- elles sont **lisibles**, donc peu abîmées par l'OCR ;
- elles couvrent **les grands sujets de l'année** : le Congo et le Katanga (phrases 1, 4, 5 et 7), les grèves (6 et 8), le mariage royal (9), le sport (2 et 10) et la culture (3) ;
- elles ont **des tons différents**. Certaines sont clairement positives (2, 3, 10), d'autres négatives (4, 6, 8) et d'autres factuelles (5, 7). Nous pouvons ainsi vérifier si les outils font la différence.

Les phrases sont recopiées telles qu'elles sont dans le corpus. La phrase 9 contient par exemple une erreur d'OCR (« mômes » au lieu de « mêmes »).

In [5]:
# Chaque phrase est notée avec la date du journal dont elle vient
phrases = [
    ("1960-02-26", "Le personnel congolais, de même que le personnel d'origine européenne manifestent un excellent esprit de collaboration."),
    ("1960-03-18", "C’est un coureur intelligent, qui sait se placer toujours d’une manière admirable à l’occasion d’un sprint massif."),
    ("1960-04-23", "Nous avions vu cette pièce présentée par les Galas Karsenty en 1954, dans une mise en scène assez fascinante de Jean Mercure et dans l’admirable décor de Wakhévitch."),
    ("1960-07-10", "Politiquement, la situation au Katanga paraît être la plus préoccupante."),
    ("1960-07-10", "Tshombe a fait, en outre, connaître son intention de proclamer l'indépendance du Katanga."),
    ("1960-07-13", "Le gouvernement a déclaré la grève Illégale et plus de 500 personnes ont été arrêtées."),
    ("1960-07-30", "Lumumba déclara plusieurs fois, en public, et jusqu’en conseil des ministres, sa volonté de tenir pleinement compte des avis des techniciens belges dont il s’entourait."),
    ("1960-09-07", "Il réprouve totalement les déclarations du Premier ministre contre le droit de grève."),
    ("1960-10-26", "Les mômes étudiants avaient donné une sérénade à Madrid, à dona Fabiola, qui leur aurait alors déclaré en les remerciant : « J’aimerais beaucoup que Baudouin vous rencontre et vous entende »."),
    ("1960-10-30", "Nous avions vécu une heure de très bon football de la part des Diables rouges."),
]
len(phrases)

10

In [6]:
# Appliquer la fonction du cours sur chaque phrase
for date, phrase in phrases:
    print(f"[{date}] {phrase}")
    get_sentiment(phrase)
    print()

[1960-02-26] Le personnel congolais, de même que le personnel d'origine européenne manifestent un excellent esprit de collaboration.
This text is 12% positive and 32% subjective.

[1960-03-18] C’est un coureur intelligent, qui sait se placer toujours d’une manière admirable à l’occasion d’un sprint massif.
This text is 53% positive and 27% subjective.

[1960-04-23] Nous avions vu cette pièce présentée par les Galas Karsenty en 1954, dans une mise en scène assez fascinante de Jean Mercure et dans l’admirable décor de Wakhévitch.
This text is 70% positive and 60% subjective.

[1960-07-10] Politiquement, la situation au Katanga paraît être la plus préoccupante.
This text is 15% positive and 10% subjective.

[1960-07-10] Tshombe a fait, en outre, connaître son intention de proclamer l'indépendance du Katanga.
This text is neutral and perfectly objective.

[1960-07-13] Le gouvernement a déclaré la grève Illégale et plus de 500 personnes ont été arrêtées.
This text is 0% negative and 3% subj

In [7]:
# Garder les scores bruts (polarité de -1 à +1, subjectivité de 0 à 1) pour construire le tableau
# Le "+ 0.0" évite l'affichage "-0.0" quand la polarité est nulle
rows = []
for date, phrase in phrases:
    polarity, subjectivity = tb(phrase).sentiment
    rows.append({"date": date, "phrase": phrase, "polarité": round(polarity, 2) + 0.0, "subjectivité": round(subjectivity, 2)})

df = pd.DataFrame(rows)
df.index = df.index + 1
pd.set_option('display.max_colwidth', None)
df

,date,phrase,polarité,subjectivité
1,1960-02-26,"Le personnel congolais, de même que le personnel d'origine européenne manifestent un excellent esprit de collaboration.",0.12,0.32
2,1960-03-18,"C’est un coureur intelligent, qui sait se placer toujours d’une manière admirable à l’occasion d’un sprint massif.",0.53,0.27
3,1960-04-23,"Nous avions vu cette pièce présentée par les Galas Karsenty en 1954, dans une mise en scène assez fascinante de Jean Mercure et dans l’admirable décor de Wakhévitch.",0.70,0.60
4,1960-07-10,"Politiquement, la situation au Katanga paraît être la plus préoccupante.",0.15,0.10
5,1960-07-10,"Tshombe a fait, en outre, connaître son intention de proclamer l'indépendance du Katanga.",0.00,0.00
6,1960-07-13,Le gouvernement a déclaré la grève Illégale et plus de 500 personnes ont été arrêtées.,0.00,0.03
7,1960-07-30,"Lumumba déclara plusieurs fois, en public, et jusqu’en conseil des ministres, sa volonté de tenir pleinement compte des avis des techniciens belges dont il s’entourait.",0.05,0.05
8,1960-09-07,Il réprouve totalement les déclarations du Premier ministre contre le droit de grève.,0.26,0.32
9,1960-10-26,"Les mômes étudiants avaient donné une sérénade à Madrid, à dona Fabiola, qui leur aurait alors déclaré en les remerciant : « J’aimerais beaucoup que Baudouin vous rencontre et vous entende ».",0.10,0.05
10,1960-10-30,Nous avions vécu une heure de très bon football de la part des Diables rouges.,0.51,0.50


**Commentaire (TextBlob).** La polarité va de -1 (très négatif) à +1 (très positif). La subjectivité va de 0 (objectif) à 1 (très subjectif).

- **Aucune phrase n'a une polarité négative.** La polarité moyenne est de 0,24. Pourtant, trois phrases sont négatives pour un lecteur humain (4, 6 et 8). TextBlob donne même +0,26 à la phrase 8 (« Il réprouve totalement les déclarations... »). Nous avons testé les mots un par un pour comprendre. L'outil ne connaît pas le verbe « réprouve » (score 0). Il se base donc sur « Premier » (+0,5), « droit » (+0,17) et « totalement » (+0,1), qu'il juge positifs. De même, il ne connaît pas « préoccupante » dans la phrase 4 : le score de +0,15 vient seulement du mot « plus ». TextBlob additionne donc des mots, mais il ne comprend pas le sens de la phrase.
- **Les phrases de sport et de culture ont les scores les plus élevés** : 0,70 pour la phrase 3, 0,53 pour la phrase 2 et 0,51 pour la phrase 10. Cela s'explique par des adjectifs clairs comme « admirable », « fascinante » ou « très bon ». TextBlob fonctionne donc bien quand le texte contient des adjectifs d'opinion.
- **Les phrases politiques sont jugées presque objectives.** Les phrases 5, 6, 7 et 9 ont une subjectivité entre 0,00 et 0,05. C'est logique, car elles rapportent des faits sans adjectif d'opinion. La subjectivité moyenne est faible (0,22) : le journal écrit de manière factuelle sur la politique.

## 2. Utilisation de transformers

Documentation: https://github.com/TheophileBlard/french-sentiment-analysis-with-bert

**!!** Si le code ne tourne pas sur votre machine, vous pouvez le tester directement sur Google Colab en utilisant [ce lien](https://colab.research.google.com/github/TheophileBlard/french-sentiment-analysis-with-bert/blob/master/colab/french_sentiment_analysis_with_bert.ipynb) **!!**

Le modèle peut également être testé en ligne sur [HuggingFace](https://huggingface.co/tblard/tf-allocine)

Note : `transformers` v5 a supprimé le support TensorFlow. On utilise donc ici `philschmid/pt-tblard-tf-allocine`, un portage PyTorch du même modèle, avec les classes `AutoModelForSequenceClassification` (au lieu de `TFAutoModelForSequenceClassification`).

### Installation des librairies et imports

In [8]:
!uv pip install sentencepiece

from transformers import AutoTokenizer, AutoModelForSequenceClassification
from transformers import pipeline

Using Python 3.13.15 environment at: C:\Users\COMPUTER STORES\Desktop\BLOC II\tac\.venv
Checked 1 package in 42ms
C:\Users\COMPUTER STORES\Desktop\BLOC II\tac\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### Chargement du modèle

In [9]:
tokenizer = AutoTokenizer.from_pretrained("philschmid/pt-tblard-tf-allocine")
model = AutoModelForSequenceClassification.from_pretrained("philschmid/pt-tblard-tf-allocine")

sentiment_analyser = pipeline('sentiment-analysis', model=model, tokenizer=tokenizer)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 10470.78it/s]

### Analyser le sentiment d'une phrase

In [10]:
sentiment_analyser("Ce journal est vraiment super intéressant.")

[{'label': 'POSITIVE', 'score': 0.985571026802063}]

In [11]:
sentiment_analyser("Cette phrase est négative et je ne suis pas content !")

[{'label': 'NEGATIVE', 'score': 0.9271845817565918}]

### Analyser les 10 phrases de 1960 avec le modèle

In [12]:
results = sentiment_analyser([phrase for date, phrase in phrases])
df["modèle BERT"] = [r["label"] for r in results]
df["confiance BERT"] = [round(r["score"], 2) for r in results]
df

,date,phrase,polarité,subjectivité,modèle BERT,confiance BERT
1,1960-02-26,"Le personnel congolais, de même que le personnel d'origine européenne manifestent un excellent esprit de collaboration.",0.12,0.32,POSITIVE,0.83
2,1960-03-18,"C’est un coureur intelligent, qui sait se placer toujours d’une manière admirable à l’occasion d’un sprint massif.",0.53,0.27,POSITIVE,0.67
3,1960-04-23,"Nous avions vu cette pièce présentée par les Galas Karsenty en 1954, dans une mise en scène assez fascinante de Jean Mercure et dans l’admirable décor de Wakhévitch.",0.70,0.60,POSITIVE,0.91
4,1960-07-10,"Politiquement, la situation au Katanga paraît être la plus préoccupante.",0.15,0.10,NEGATIVE,0.70
5,1960-07-10,"Tshombe a fait, en outre, connaître son intention de proclamer l'indépendance du Katanga.",0.00,0.00,POSITIVE,0.50
6,1960-07-13,Le gouvernement a déclaré la grève Illégale et plus de 500 personnes ont été arrêtées.,0.00,0.03,NEGATIVE,0.59
7,1960-07-30,"Lumumba déclara plusieurs fois, en public, et jusqu’en conseil des ministres, sa volonté de tenir pleinement compte des avis des techniciens belges dont il s’entourait.",0.05,0.05,POSITIVE,0.70
8,1960-09-07,Il réprouve totalement les déclarations du Premier ministre contre le droit de grève.,0.26,0.32,NEGATIVE,0.65
9,1960-10-26,"Les mômes étudiants avaient donné une sérénade à Madrid, à dona Fabiola, qui leur aurait alors déclaré en les remerciant : « J’aimerais beaucoup que Baudouin vous rencontre et vous entende ».",0.10,0.05,POSITIVE,0.62
10,1960-10-30,Nous avions vécu une heure de très bon football de la part des Diables rouges.,0.51,0.50,POSITIVE,0.66


**Commentaire (modèle BERT).** Le modèle donne seulement deux étiquettes : POSITIVE ou NEGATIVE. Il n'a pas de catégorie neutre. La colonne « confiance » indique à quel point il est sûr de sa réponse (de 0,5 à 1).

- Le modèle classe correctement les trois phrases négatives (4, 6 et 8), alors que TextBlob les avait manquées. Il comprend donc mieux le sens général de la phrase.
- En revanche, sa confiance est souvent faible. Elle est de 0,50 pour la phrase 5, ce qui revient à un tirage au sort. Cette phrase est en réalité neutre (« Tshombe a fait connaître son intention... »), mais le modèle est obligé de choisir.
- La confiance est la plus forte pour la phrase 3 (0,91), qui parle d'une pièce de théâtre. Ce résultat est logique, car le modèle a appris sur des critiques de films. Il est donc plus à l'aise avec le vocabulaire du spectacle qu'avec celui de la politique.

### Comparer les deux outils

In [13]:
# Traduire la polarité de TextBlob en étiquette pour la comparer à celle du modèle
def label_textblob(polarity):
    if polarity > 0:
        return "POSITIVE"
    elif polarity < 0:
        return "NEGATIVE"
    return "NEUTRE"

df["TextBlob"] = df["polarité"].apply(label_textblob)
df["accord"] = df["TextBlob"] == df["modèle BERT"]
print(df[["date", "polarité", "subjectivité", "TextBlob", "modèle BERT", "confiance BERT", "accord"]].to_string())
print()
print(f"Les deux outils sont d'accord sur {df['accord'].sum()} phrases sur {len(df)}.")
print(f"Polarité moyenne : {df['polarité'].mean():.2f} - Subjectivité moyenne : {df['subjectivité'].mean():.2f}")

          date  polarité  subjectivité  TextBlob modèle BERT  confiance BERT  accord
1   1960-02-26      0.12          0.32  POSITIVE    POSITIVE            0.83    True
2   1960-03-18      0.53          0.27  POSITIVE    POSITIVE            0.67    True
3   1960-04-23      0.70          0.60  POSITIVE    POSITIVE            0.91    True
4   1960-07-10      0.15          0.10  POSITIVE    NEGATIVE            0.70   False
5   1960-07-10      0.00          0.00    NEUTRE    POSITIVE            0.50   False
6   1960-07-13      0.00          0.03    NEUTRE    NEGATIVE            0.59   False
7   1960-07-30      0.05          0.05  POSITIVE    POSITIVE            0.70    True
8   1960-09-07      0.26          0.32  POSITIVE    NEGATIVE            0.65   False
9   1960-10-26      0.10          0.05  POSITIVE    POSITIVE            0.62    True
10  1960-10-30      0.51          0.50  POSITIVE    POSITIVE            0.66    True

Les deux outils sont d'accord sur 6 phrases sur 10.
Polarité moy

**Commentaire (comparaison).** Les deux outils sont d'accord sur 6 phrases sur 10. Les quatre désaccords concernent tous des phrases politiques (4, 5, 6 et 8). Les six accords concernent surtout le sport, la culture et les phrases au ton positif.

Nous en tirons trois conclusions :
1. Les outils sont fiables quand la phrase contient des mots d'opinion clairs. C'est le cas dans les pages de sport et de culture.
2. Ils sont moins fiables sur les phrases politiques, car le journal y décrit des faits graves avec des mots neutres. La phrase 6 annonce 500 arrestations, mais TextBlob lui donne une polarité de 0.
3. Aucun des deux outils n'a été conçu pour la presse de 1960. TextBlob repose sur un lexique moderne et BERT sur des critiques de films. Il faut donc lire ces résultats avec prudence, surtout sur un texte ancien et abîmé par l'OCR.